## Data Quality

O código apresentado neste notebook foi desenvolvido durante a análise da qualidade da base de dados e tem como objetivo compreender possíveis regras de negócio e como elas estão representadas na base de dados, além de identificar inconsistências, problemas de tipagem, valores ausentes, duplicatas e outros possíveis problemas de qualidade.

In [1]:
import pandas as pd

from pathlib import Path

In [2]:
df = pd.read_csv("../dataset/dataset_limpo_univariado.csv")

### Dados pessoais

Há alguns fatores principais que podem ser usados como referência para uma investigação de justiça algorítmica, na nossa base de dados eles são: 

- Idade
- Grau de instrução escolar
- Status civil
- Status econômico (tendo como proxy residência)
- Sexo biológico 

Abaixo vamos explorar de forma multivariada se possuímos dados suficientes para investigarmos situações de (in)justiça algorítimica com base nesses tópicos, se há alguma inconsistência e explorarmos possíveis padrões nos dados. 

In [3]:
COLUNAS_ESCOLARIDADE = [
    "QT_CLI_ANF",
    "QT_CLI_PRI_GR",
    "QT_CLI_SEG_GR",
    "QT_CLI_SUP"
]

COLUNAS_GENERO = [
    'QT_CLI_MLH', 
    'QT_CLI_HOM'
]

COLUNAS_STATUS_CIVIL = [
    "QT_CLI_STR",
    "QT_CLI_CAS",
    "QT_CLI_SPD"
]

COLUNAS_RESIDENCIAS = [
    "QT_RES_PRP",
    "QT_RES_ALU",
    "QT_RES_OUT"
]

- Análise por gênero e quantidade total de clientes

In [4]:
soma_genero = df[COLUNAS_GENERO].fillna(0).sum(axis=1) # Calcula a soma de sexo preenchendo nulos com 0

registros_consistentes = (soma_genero == df['QT_CLI']).sum()
registros_inconsistentes = (soma_genero != df['QT_CLI']).sum()
total = len(df)

print(f"Total de registros: {total}")
print(f"Registros consistentes: {registros_consistentes} ({(registros_consistentes/total)*100:.2f}%)")
print(f"Registros Inconsistentes: {registros_inconsistentes} ({(registros_inconsistentes/total)*100:.2f}%)")

Total de registros: 316053
Registros consistentes: 316053 (100.00%)
Registros Inconsistentes: 0 (0.00%)


Quis investigar se os dados se mostram consistentes no que diz respeito a quantidades registradas. Inicialmente vemos que as colunas que dizem respeito a quantidade de homens e quantidade de mulheres no grupo é compatível com a quantidade total de clientes que está registrada no grupo em 100% da base de dados. 

- Análise por nivel de escolaridade e quantidade total de clientes

In [5]:
soma_dados_escolaridade = df[COLUNAS_ESCOLARIDADE].fillna(0).sum(axis=1)

registros_consistentes = (soma_dados_escolaridade == df['QT_CLI']).sum()
registros_inconsistentes = (soma_dados_escolaridade < df['QT_CLI']).sum()
total = len(df)

print(f"Total de registros: {total}")
print(f"Registros consistentes: {registros_consistentes} ({(registros_consistentes/total)*100:.2f}%)")
print(f"Registros Inconsistentes: {registros_inconsistentes} ({(registros_inconsistentes/total)*100:.2f}%)")

Total de registros: 316053
Registros consistentes: 63 (0.02%)
Registros Inconsistentes: 315990 (99.98%)


Outra feature necessária de se investigar sua consistência é a de escolaridade dos clientes. Em 99.98% da base de dados a quantidade de clientes que indicou seu nível de escolaridade é menor que a quantidade de clientes, tornando assim, **uma feature fragil para uso como parâmetro de investigação de justiça algorítmica, sendo necessário possivelmente seu descarte.**

- Análise por status civil e quantidade de clientes

In [6]:
soma_status_civil = df[COLUNAS_STATUS_CIVIL].fillna(0).sum(axis=1)

registros_consistentes = (soma_status_civil == df['QT_CLI']).sum()
registros_inconsistentes = (soma_status_civil < df['QT_CLI']).sum()
total = len(df)

print(f"Total de registros: {total}")
print(f"Registros consistentes: {registros_consistentes} ({(registros_consistentes/total)*100:.2f}%)")
print(f"Registros Inconsistentes: {registros_inconsistentes} ({(registros_inconsistentes/total)*100:.2f}%)")

Total de registros: 316053
Registros consistentes: 309528 (97.94%)
Registros Inconsistentes: 6525 (2.06%)


No que diz respeito a informações sobre status civil, temos que há 97.9% da base com registros consistentes, ou seja, que todos os clintes do grupo indicaram seu status civil. Acredito que o melhor caminho seria manter apenas os registros consistentes. Isso é feito na célula abaixo:

In [7]:
registros_consistentes = soma_status_civil == df['QT_CLI']
df = df[registros_consistentes].reset_index(drop=True)

Por fim, verificamos a qualidade de dados referente as colunas de residências. Essas colunas indicam, a respeito da moradia dos clientes, quantas residências são próprias, quantas residências são alugadas, quantas residências são emprestadas ou mora com parentescos.

- Analise por quantidade de residências e quantidade de clientes

In [8]:
soma_residencias = df[COLUNAS_RESIDENCIAS].fillna(0).sum(axis=1)

registros_um_pra_um = (soma_residencias == df['QT_CLI']).sum() # Supõe-se que cada cliente está relacionado a uma, e apenas uma, moradia
registros_maior_que_um = (soma_residencias > df['QT_CLI']).sum() # Há mais de uma moradia por cliente do grupo
registros_menor_que_um = (soma_residencias < df['QT_CLI']).sum() # Há menos de uma moradia por cliente de dentro do grupo
total = len(df)

print(f"Total de registros: {total}")
print(f"Registros uma moradia por cliente do grupo: {registros_um_pra_um} ({(registros_um_pra_um/total)*100:.2f}%)")
print(f"Registros mais de uma moradia por cliente do grupo {registros_maior_que_um} ({(registros_maior_que_um/total)*100:.2f}%)")
print(f"Registros menos de uma moradia por cliente de dentro do grupo: {registros_menor_que_um} ({(registros_menor_que_um/total)*100:.2f}%)")

Total de registros: 309528
Registros uma moradia por cliente do grupo: 281536 (90.96%)
Registros mais de uma moradia por cliente do grupo 0 (0.00%)
Registros menos de uma moradia por cliente de dentro do grupo: 27992 (9.04%)


Apesar de não haver grupos de clientes com mais de uma moradia por pessoa, há cerca de 9% da base de dados com menos de uma moradia por cliente. Visto que as opções foram (1) residência própria, (2) residência alugada e (3) residência emprestada/mora com parentescos, o que isso quer dizer? que há clientes que não possuem moradia/moram na rua? Não faz sentido, só pode ter sido erro de registro/coleta de dados. Vamos remover esses registros momentaneamente.

In [9]:
registros_consistentes = soma_residencias == df['QT_CLI']
df = df[registros_consistentes].reset_index(drop=True)

### Dados de proposta

In [10]:
total = len(df)
valor_aprovado_maior = (df['VR_APR_PRP'] > df['VR_SOL_PRP']).sum() # Supõe-se que cada cliente está relacionado a uma, e apenas uma, moradia
valor_aprovado_menor = (df['VR_APR_PRP'] < df['VR_SOL_PRP']).sum() # Há mais de uma moradia por cliente do grupo
valor_aprovado_igual = (df['VR_APR_PRP'] == df['VR_SOL_PRP']).sum() # Há menos de uma moradia por cliente de dentro do grupo

print(f"Há {total} propostas")
print(f"{valor_aprovado_maior} tiveram um valor aprovado maior do que o solicitado ({(valor_aprovado_maior/total)*100:.2f}%)")
print(f"{valor_aprovado_menor} tiveram um valor aprovado menor do que o solicitado ({(valor_aprovado_menor/total)*100:.2f}%)")
print(f"{valor_aprovado_igual} tiverem um valor aprovado igual ao solicitado ({(valor_aprovado_igual/total)*100:.2f}%)")

Há 281536 propostas
2590 tiveram um valor aprovado maior do que o solicitado (0.92%)
140215 tiveram um valor aprovado menor do que o solicitado (49.80%)
138731 tiverem um valor aprovado igual ao solicitado (49.28%)


Após a remoção ficamos com um total de 281 mil 536 registros.

### Dados de pesquisa cadastrais

In [ ]:
total_registros = len(df)
registros_inconsistentes_spc = (df['QT_CLI_SPC'] > df['QT_CLI']).sum()
registros_inconsistentes_serasa = (df['QT_CLI_SRS'] > df['QT_CLI']).sum()
registros_inconsistentes_bnb = (df['QT_CLI_BNB'] > df['QT_CLI']).sum()

print(f"Total de registros: {total_registros}")
print(f"Há {registros_inconsistentes_spc} registros inconsistentes a respeito do SPC - Mais clientes no SPC do que no grupo.")
print(f"Há {registros_inconsistentes_serasa} registros inconsistentes a respeito do SERASA - Mais clientes no SERASA do que no grupo.")
print(f"Há {registros_inconsistentes_bnb} registros inconsistentes a respeito do BNB - Mais clientes no BNB do que no grupo.")

Total de registros: 281536
Há 1723 registros inconsistentes a respeito do SPC. Mais clientes no SPC do que no grupo.
Há 468 registros inconsistentes a respeito do SERASA. Mais clientes no SERASA do que no grupo.
Há 18 registros inconsistentes a respeito do BNB. Mais clientes no BNB do que no grupo.


In [17]:
registros_para_remover = (df['QT_CLI_SPC'] > df['QT_CLI']) | (df['QT_CLI_SRS'] > df['QT_CLI']) | (df['QT_CLI_BNB'] > df['QT_CLI'])

df_final = df.loc[~registros_para_remover]

In [18]:
filepath = Path("../dataset/dataset_limpo_multivariado.csv")
df_final.to_csv(filepath)